# Comparação entre SARIMA e SARIMAX

Este caderno compara ajuste e desempenho preditivo de SARIMA e SARIMAX para as 27 UFs e o Brasil, nos recortes pré-pandemia e completo.

Os SARIMAX individuais acrescentam uma tipologia por vez; os gerais reúnem as tipologias admitidas e suas defasagens. São examinadas as grades de 0 a K e de 3 a K, até 24 meses. O BIC é avaliado no treino e o RMSE no teste cronológico. A resposta é 100Δlog(I) e a exposição é Δlog(1+D).

As taxas nacionais são calculadas pelos saldos agregados antes das transformações. O [guia metodológico](../docs/metodologia_e_sensibilidades.md) detalha a seleção e a avaliação condicional de um passo.

In [1]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'outputs').exists(): ROOT=ROOT.parent
B=ROOT/'outputs/tables/modalidades_excluidas_20261104'
import json
manifest=json.loads((B/'manifesto.json').read_text(encoding='utf-8'))
print('Base PF com três exclusões:', manifest['base_sha256'])
print('Período:', manifest['periodo'], 'Observações:', manifest['linhas'])


Base PF com três exclusões: d795e0ed176cc4628e1b02a81bc32ecab449f24aefc3e9be91d291479f46c043
Período: ['2013-01', '2024-12'] Observações: 3888


## Comparação com o SARIMA de referência

BIC menor descreve ajuste penalizado no treino. RMSE menor descreve a previsão condicional no teste. Ganhos nos dois critérios são contados em separado.

In [2]:
rows=[]
national=[]
for lag in [0,3]:
    r=pd.read_csv(B/f'sarimax{lag}/resumo.csv')
    g=pd.read_csv(B/f'sarimax{lag}/geral.csv')
    assert len(r)==896 and len(g)==56
    rows.append(dict(grade=f'{lag}..K',individuais_estimados=int(r.status.eq('estimado').sum()),BIC_menor=int(r.delta_BIC.lt(0).sum()),RMSE_menor=int(r.melhora_rmse_pct.gt(0).sum()),ambos=int((r.delta_BIC.lt(0)&r.melhora_rmse_pct.gt(0)).sum()),gerais_estimados=int(g.status.eq('estimado').sum()),gerais_ambos=int((g.delta_BIC.lt(0)&g.melhora_rmse_pct.gt(0)).sum())))
    national.append(g[g.unidade.eq('Brasil')].assign(grade=f'{lag}..K'))
print(pd.DataFrame(rows).to_string(index=False))

grade  individuais_estimados  BIC_menor  RMSE_menor  ambos  gerais_estimados  gerais_ambos
 0..K                    452         22         173      5                54             0
 3..K                    452         29         170      6                55             1


## Defasagens selecionadas

O menor K permitido corresponde a um único coeficiente de exposição em cada grade. Sua predominância não demonstra o instante de início nem a duração de um efeito; K=24 é o limite superior da busca.

In [3]:
h=[]
for lag in [0,3]:
    r=pd.read_csv(B/f'sarimax{lag}/resumo.csv')
    h.append(r[r.status.eq('estimado')].assign(grade=f'{lag}..K'))
print(pd.concat(h).groupby(['grade','K']).size().rename('modelos').to_string())

grade  K   
0..K   0.0     427
       1.0      14
       2.0       3
       3.0       2
       4.0       2
       6.0       1
       7.0       1
       17.0      1
       24.0      1
3..K   3.0     424
       4.0      17
       5.0       2
       6.0       2
       7.0       3
       8.0       1
       9.0       1
       14.0      1
       17.0      1


## Agregado Brasil

ΔBIC é BIC(SARIMAX) − BIC(SARIMA): valores negativos favorecem o SARIMAX. A melhora percentual no RMSE é positiva quando o erro diminui.

In [4]:
print(pd.concat(national)[['grade','periodo','n_tipologias_incluidas','delta_BIC','melhora_rmse_pct','lb12_padronizado']].to_string(index=False))

grade periodo  n_tipologias_incluidas  delta_BIC  melhora_rmse_pct  lb12_padronizado
 0..K     Pré                      16  48.566543        -40.604273          0.000036
 0..K   Total                      16  62.056391         -8.968644          0.178551
 3..K     Pré                      16  53.361953          3.037970          0.001134
 3..K   Total                      16  53.948917        -34.961035          0.255191


## Comparação pareada entre as duas grades

Aqui a referência é o SARIMAX 0..K, e não o SARIMA. As duas versões foram conferidas para utilizar as mesmas amostras, ordens e métricas do modelo de referência.

In [5]:
p=pd.read_csv(B/'sarimax3/comparacao_individuais.csv'); ok=p[p.ambos_estimaveis]; print(ok.groupby('periodo').agg(pares=('unidade','size'),BIC_melhor_3aK=('BIC_melhor_3aK','sum'),RMSE_melhor_3aK=('RMSE_melhor_3aK','sum')).to_string())

         pares  BIC_melhor_3aK  RMSE_melhor_3aK
periodo                                        
Pré        194              97              105
Total      258             149              119


## Integridade e limites

Os 1.792 casos individuais e os 112 modelos gerais planejados nas duas grades foram conferidos quanto à cobertura, escolha de K pelo BIC entre candidatos convergentes, amostra comum e cálculo das métricas. Aplicações não estimáveis permanecem identificadas.

In [6]:
v=pd.read_csv(B/'validacao_sarimax.csv'); assert len(v)==112 and v.passou.all(); print('Combinações de grade/unidade/período conferidas:',len(v)); print(pd.read_csv(B/'validacao_nacional_sarimax.csv').to_string(index=False))

Combinações de grade/unidade/período conferidas: 112
periodo   n  max_erro_y  max_erro_x
    Pré  84         0.0         0.0
  Total 143         0.0         0.0


A inferência é condicional à seleção das ordens e lags. Quando lag0 é admitido, a previsão é condicional à exposição realizada do mês, não uma previsão ex ante sem essa informação. A seleção concentrada no menor K permitido também pode refletir a penalização do BIC. Ganhos pontuais não identificam impacto causal nem uma tipologia de maior efeito.